# OTFLEX Gavin Workflow (Operation-Based Notebook)

This notebook executes the full experimental sequence directly in notebook cells, without graph traversal.

Execution model here:
- Each workflow action is a separate runnable code cell.
- Parameters are declared at the top of each action cell for easy edits.
- Cells are order-agnostic and can be rearranged.
- If something fails, jump to the teardown cell to leave devices in a safe state.

In [3]:
import asyncio
import json
import subprocess
import sys
import time
from pathlib import Path

# Find repo root so imports from src.* work no matter where notebook launches from.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not find repository root containing src/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.adapters.iot_mqtt import (
    PumpMQTT,
    UltraMQTT,
    HeatMQTT,
    ReactorMQTT,
    FurnaceMQTT,
    start_broker_if_needed,
    stop_broker,
    ControllerBeacon,
    _best_effort_all_off,
)
from src.adapters.otflex_adapter import OTFlex
from src.core.opentrons import opentronsClient

print(f"Repo root: {repo_root}")

# Device configs defined directly in this notebook
otflex_cfg = {
    "module": "otflex_runtime.py",
    "controller_ip": "192.168.10.161", # might be on 192.168.10.161, 169.254.179.32
    "deck": {
        "slots": {
            "A1": {"name": "electrode_module", "display": "3 electrodes", "labware": "sdl1_single_electrode_tiprack", "file": "data/labware_definitions/sdl1_single_electrode_tiprack.json", "slot_label": "A1"},
            "A2": {"name": "parallel_electrode_module", "display": "4 electrode tool", "labware": "sdl1_parallel_electrode_tiprack", "file": "data/labware_definitions/sdl1_parallel_electrode_tiprack.json", "slot_label": "A2"},
            "A3": {"name": "trash", "display": "trash", "labware": "opentrons_flex_trash", "slot_label": "A3"},
            "B1": {"name": "auto_reactor", "display": "auto_reactor", "labware": "actuated_reactor", "file": "data/labware_definitions/actuated_reactor.json", "slot_label": "B1"},
            "B2": {"name": "source_plate", "display": "precursor solutions", "labware": "sdl1_11_vials_20mL", "file": "data/labware_definitions/sdl1_11_vials_20mL.json", "slot_label": "B2"},
            "B3": {"name": "sonicator_bath", "display": "sonicator bath", "labware": "nis_2_sonicator_bath", "file": "data/labware_definitions/nis_2_sonicator_bath.json", "slot_label": "B3"},
            "C3": {"name": "tiprack_1000ul", "display": "1000ul tips", "labware": "opentrons_flex_96_tiprack_1000ul", "slot_label": "C3"},
            "D1": {"name": "substrate_tower", "display": "tower of substrates", "labware": "zou_21_wellplate_4500ul", "file": "data/labware_definitions/zou_21_wellplate_4500ul.json", "slot_label": "D1"}
        },
        "pipettes": {"right": {"model": "p1000_single_flex"}}
    }
}

mqtt_cfg = {
    "broker": "192.168.0.100",
    "port": 1883,
    "username": "pyctl-controller",
    "password": "controller",
    "topics": {
        "pumps": "pumps/01",
        "reactor": "reactor/01",
        "furnace": "furnace/01",
        "heat": "heat/01",
        # Keep aligned with MQTT_Demo UltraMQTT base_topic.
        "ultra": "ultra/01",
    },
}

# Critical: flushWell uses OTFlex runtime MQTT helpers, so pass MQTT config into otflex runtime.
otflex_cfg["mqtt"] = mqtt_cfg

opentrons_cfg = {
    "ip": otflex_cfg["controller_ip"],
    "robot": "flex",
}

# Shared runtime objects.
otflex = OTFlex(otflex_cfg, root_dir=repo_root)

beacon = None
pumps = None
ultra = None
heat = None
reactor = None
furnace = None

print("Setup objects created. Run homing cell next, then connect devices.")

Repo root: C:\Users\Dell PC\Desktop\Projects\AC-OTFlex-monorepo
[OTFlex] Loaded module: C:\Users\Dell PC\Desktop\Projects\AC-OTFlex-monorepo\src\core\otflex_runtime.py
Setup objects created. Run homing cell next, then connect devices.


## Home Opentrons Robot First

Run this before any workflow action.

This cell inlines the same approach used by `scripts/home_opentrons.py` (directly using `opentronsClient`), without importing that script.

In [2]:
# Homing params (editable)
homing_params = {
    "ip": opentrons_cfg["ip"],
    "robot": opentrons_cfg["robot"],
}

client = opentronsClient(
    strRobotIP=homing_params["ip"],
    strRobot=homing_params["robot"],
)
client.homeRobot()
print(f"Homed {homing_params['robot']} at {homing_params['ip']}")

Homed flex at 192.168.10.161


## Capture Lab Setup Image (Before Connect)

Run this after homing and before connecting devices.

This uses the same SSH capture flow as the verbose Pi camera notebook:
- connect to Raspberry Pi over SSH
- capture an image with Picamera2
- download to `data/out/images`
- rotate the image 180 degrees
- display the image inline
- remove temporary remote image

In [ ]:
# Camera params (editable)
camera_params = {
    "host": "192.168.0.101",
    "username": "sdl1",
    "password": "1144",
    "ssh_port": 22,
    "connect_timeout_s": 8,
    "connect_retries": 3,
    "remote_capture_dir": "/tmp",
    "warmup_s": 2,
    "rotate_degrees": 180,
}

from datetime import datetime
import socket

try:
    import paramiko
except Exception as exc:
    raise ImportError(
        "paramiko is required for SSH camera capture. Install it in the notebook environment."
    ) from exc

try:
    from PIL import Image as PILImage
    from IPython.display import Image as IPyImage, display
except Exception as exc:
    raise ImportError(
        "Pillow and IPython display are required for rotate/display. Install pillow in the notebook environment."
    ) from exc

out_dir = repo_root / "data" / "out" / "images"
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"otflex-top_{timestamp}.jpg"
remote_path = f"{camera_params['remote_capture_dir']}/{filename}"
local_path = out_dir / filename

with socket.create_connection(
    (camera_params["host"], int(camera_params["ssh_port"])),
    timeout=3,
    ):
    pass
print(f"SSH port reachable at {camera_params['host']}:{camera_params['ssh_port']}")

ssh = None
try:
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    for attempt in range(1, int(camera_params["connect_retries"]) + 1):
        try:
            print(f"SSH connect attempt {attempt}/{camera_params['connect_retries']}")
            ssh.connect(
                camera_params["host"],
                port=int(camera_params["ssh_port"]),
                username=camera_params["username"],
                password=camera_params["password"],
                timeout=float(camera_params["connect_timeout_s"]),
                banner_timeout=float(camera_params["connect_timeout_s"]),
                auth_timeout=float(camera_params["connect_timeout_s"]),
            )
            break
        except (socket.timeout, TimeoutError):
            if attempt == int(camera_params["connect_retries"]):
                raise
            print("Timeout; retrying...")

    capture_cmd = f"""python3 << 'EOF'
from picamera2 import Picamera2
import time

picam2 = Picamera2()
config = picam2.create_still_configuration(main={{\"size\": (2028, 1520)}})
picam2.configure(config)
picam2.start()
time.sleep({float(camera_params['warmup_s'])})
picam2.capture_file(\"{remote_path}\")
picam2.close()
print(\"OK\")
EOF
"""

    _, stdout, stderr = ssh.exec_command(capture_cmd, timeout=45)
    exit_code = stdout.channel.recv_exit_status()
    err_text = stderr.read().decode()
    if exit_code != 0:
        raise RuntimeError(f"Pi capture failed: {err_text}")

    sftp = ssh.open_sftp()
    sftp.get(remote_path, str(local_path))
    sftp.close()
    ssh.exec_command(f"rm {remote_path}")

    print(f"Lab setup image saved: {local_path}")

    # Rotate image to match top-down orientation used in verbose notebook.
    with PILImage.open(local_path) as img:
        rotated = img.rotate(int(camera_params["rotate_degrees"]), expand=True)
        rotated.save(local_path)

    print(f"Showing rotated image: {local_path.name}")
    display(IPyImage(filename=str(local_path)))
except Exception:
    raise
finally:
    if ssh is not None:
        ssh.close()

## Connect Devices

This cell mirrors MQTT_Demo MQTT client setup as closely as possible:
- connect OTFlex
- start controller beacon
- connect MQTT devices used by this workflow (pump/ultra/heat/reactor/furnace)
- use `pyctl-*` client IDs for consistent broker/client behavior

In [5]:
# Connection params (editable)
import socket
import subprocess
import time

def _mqtt_port_open(host, port, timeout=2):
    try:
        with socket.create_connection((host, int(port)), timeout=timeout):
            return True
    except OSError:
        return False

broker_host = mqtt_cfg["broker"]
broker_port = int(mqtt_cfg["port"])

if not _mqtt_port_open(broker_host, broker_port):
    print(f"MQTT broker not reachable at {broker_host}:{broker_port}; starting local broker...")
    subprocess.Popen(
        [str(repo_root / "scripts" / "start_broker.bat")],
        cwd=str(repo_root),
        creationflags=subprocess.CREATE_NEW_CONSOLE,
    )
    time.sleep(3)

if not _mqtt_port_open(broker_host, broker_port):
    raise RuntimeError(f"MQTT broker still not reachable at {broker_host}:{broker_port}")

print(f"[OK] MQTT broker reachable at {broker_host}:{broker_port}")

connect_params = {
    "settle_s": 0.8,
    "reset_existing_clients": True,
}

# Keep MQTT settings aligned with MQTT_Demo known-good values.
mqtt_cfg["broker"] = "192.168.0.100"
mqtt_cfg["port"] = 1883
mqtt_cfg["username"] = "pyctl-controller"
mqtt_cfg["password"] = "controller"
mqtt_cfg["topics"]["ultra"] = "ultra/01"

if connect_params["reset_existing_clients"]:
    # Re-running this cell with identical client IDs can cause broker-side disconnect thrash.
    for dev in (pumps, ultra, heat, reactor, furnace):
        if dev is not None:
            try:
                dev.disconnect()
            except Exception:
                pass
    if beacon is not None:
        try:
            beacon.stop()
        except Exception:
            pass

await otflex.connect()

beacon = ControllerBeacon(
    broker=mqtt_cfg["broker"],
    port=int(mqtt_cfg["port"]),
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
    client_id="pyctl-controller",
    status_topic="pyctl/status",
    heartbeat_topic="pyctl/heartbeat",
    heartbeat_interval=5.0,
    keepalive=30,
 )
beacon.start()

common = dict(
    broker=mqtt_cfg["broker"],
    port=int(mqtt_cfg["port"]),
    username=mqtt_cfg["username"],
    password=mqtt_cfg["password"],
)

pumps = PumpMQTT(**common, base_topic=mqtt_cfg["topics"]["pumps"], client_id="pyctl-pumps")
ultra = UltraMQTT(**common, base_topic=mqtt_cfg["topics"]["ultra"], client_id="pyctl-ultra")
heat = HeatMQTT(**common, base_topic=mqtt_cfg["topics"]["heat"], client_id="pyctl-heat")
reactor = ReactorMQTT(**common, base_topic=mqtt_cfg["topics"]["reactor"], client_id="pyctl-reactor")
furnace = FurnaceMQTT(**common, base_topic=mqtt_cfg["topics"]["furnace"], client_id="pyctl-furnace")

pumps.ensure_connected()
ultra.ensure_connected()
heat.ensure_connected()
reactor.ensure_connected()
furnace.ensure_connected()

# Ensure flushWell and other OTFlex MQTT helpers use known-good notebook clients.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is not None:
    rt.mqtt_pumps = pumps
    rt.mqtt_reactor = reactor
    rt.mqtt_furnace = furnace
    print("Bound OTFlex runtime MQTT handles to notebook MQTT clients.")

time.sleep(float(connect_params["settle_s"]))
print(f"MQTT connected: ultra client={ultra.client_id}, base={ultra.base}")
print("All configured devices connected.")

MQTT broker not reachable at 192.168.0.100:1883; starting local broker...
[OK] MQTT broker reachable at 192.168.0.100:1883
[OTFlex] Deck layout (normalized):
  slot A1 -> A1 : sdl1_single_electrode_tiprack (electrode_module)
  slot A2 -> A2 : sdl1_parallel_electrode_tiprack (parallel_electrode_module)
  slot A3 -> A3 : opentrons_flex_trash (trash)
  slot B1 -> B1 : actuated_reactor (auto_reactor)
  slot B2 -> B2 : sdl1_11_vials_20mL (source_plate)
  slot B3 -> B3 : nis_2_sonicator_bath (sonicator_bath)
  slot C3 -> C3 : opentrons_flex_96_tiprack_1000ul (tiprack_1000ul)
  slot D1 -> D1 : zou_21_wellplate_4500ul (substrate_tower)
[otflex-pumps] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-ultra] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-pumps] Connected to 192.168.0.100:1883
[otflex-heat] Connecting to 192.168.0.100:1883 (attempt 1)...
[otflex-ultra] Connected to 192.168.0.100:1883
[otflex-heat] Connected to 192.168.0.100:1883
[otflex-reactor] Connecting to 192.1

## Raise Autoreactor (mqttReactor)

Publishes `FORWARD:5000` to the reactor topic and waits for firmware auto-stop timing.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Run xArm Plate-to-Reactor Transfer

This action replaces the fixed pause.

The notebook waits until `src/core/plate2reactor.py` completes, then the next action can lower the reactor.

In [ ]:
# Action params (editable)
params = {
    "script_relpath": "src/core/plate2reactor.py",
    "python_exe": sys.executable,
    "timeout_s": 0,  # 0 means no timeout
}

script_path = (repo_root / params["script_relpath"]).resolve()
if not script_path.exists():
    raise FileNotFoundError(f"xArm script not found: {script_path}")

cmd = [params["python_exe"], str(script_path)]
print("Running xArm transfer script:", " ".join(cmd))

run_kwargs = dict(
    cwd=str(repo_root),
    text=True,
    capture_output=True,
    check=False,
    timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
)

try:
    result = subprocess.run(cmd, **run_kwargs)
except subprocess.TimeoutExpired as exc:
    raise TimeoutError(
        f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
    ) from exc

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"xArm transfer failed with exit code {result.returncode}: {script_path}"
    )

print("xArm transfer action complete.")

## Lower Autoreactor (mqttReactor)

Publishes `REVERSE:8000` and waits for duration alignment.

In [ ]:
# Action params (editable)
params = {
    "direction": "reverse",
    "duration_ms": 8500,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor lower action complete.")

## Transfer Precursor A1 (otflexTransfer)

Transfers precursor from `source_plate.A1` to reactor wells `A1` then `B1`.

***Note: If running a batch, refer to the Kevin_24well files***

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "source_plate", "well": "A1", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "auto_reactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## Transfer Precursor A2 (otflexTransfer)

Transfers precursor from `source_plate.A2` to reactor wells `A1` then `B1`.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "source_plate", "well": "A2", "offsetX": 0, "offsetY": 0, "offsetZ": 5},
    "to": {"labware": "auto_reactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 20},
    "volume_uL": 500,
    "move_speed": 120,
    "pipette": "p1000_single_flex",
    "tiprack": "tiprack_1000ul",
    "autopick_tip": True,
}

await otflex.transfer(params)
print("Transfer action complete.")

## ELECTRODEPOSITION - Operation Context

Initializes shared runtime handles and state used by the electrodeposition operations below.

Run this cell once before running any electrodeposition operation cells.

In [6]:
# Electrochem operation context (shared state + runtime handles)
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

if "ec_state" not in globals():
    ec_state = {
        "tool_attached": False,
        "current_well": None,
        "at_ultrasonic": False,
    }

# Default locations used by operation cells; edit as needed.
ec_defaults = {
    "source_labware": "electrode_module",
    "source_well": "A1",
    "reactor_labware": "auto_reactor",
    "ultra_slot": "B3",
    "sonicator_labware": "sonicator_bath",
    "sonicator_well_a": "A1",
    "sonicator_well_b": "A2",
    "sonicator_immersion_mm": 35.0,
    "pipette": "p1000_single_flex",
}

print("Electrochem operation context ready.")
print(f"State: {ec_state}")

Electrochem operation context ready.
State: {'tool_attached': False, 'current_well': None, 'at_ultrasonic': False}


## ELECTRODEPOSITION - Pick Up Electrode Tool

Picks the electrode tool from its storage location and marks the system as tool-attached.

Use this before any well placement, deposition, or ultrasonic cleaning operations.

In [ ]:
# Pick electrode tool from source rack
params = {
    "source_labware": ec_defaults["source_labware"],
    "source_well": ec_defaults["source_well"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "offsetZ": 0.0,
    "pipette": ec_defaults["pipette"],
}

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

src_lw = rt.lw_ids.get(params["source_labware"], params["source_labware"])

await asyncio.to_thread(
    rt.oc.pickUpTip,
    strLabwareName=src_lw,
    strPipetteName=params["pipette"],
    strWellName=params["source_well"],
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=float(params["offsetZ"]),
)

ec_state["tool_attached"] = True
ec_state["current_well"] = None
ec_state["at_ultrasonic"] = False

print(f"Tool picked from {params['source_labware']}.{params['source_well']}")

## ELECTRODEPOSITION - Move Tool To Reactor Well

Moves the attached electrode tool to the selected reactor well and positions it for in-well processing.

Use this whenever changing deposition target wells.

In [ ]:
# Move attached electrode tool into a reactor well
params = {
    "reactor_labware": ec_defaults["reactor_labware"],
    "target_well": "C4",
    "offsetX": 0.0,
    "offsetY": 0.0,
    "descent_at_well_mm": 25.0,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

dst_lw = rt.lw_ids.get(params["reactor_labware"], params["reactor_labware"])
descent_mm = float(params["descent_at_well_mm"])
if descent_mm < 0:
    raise ValueError("descent_at_well_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=dst_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=dst_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-descent_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["current_well"] = params["target_well"]
ec_state["at_ultrasonic"] = False
print(f"Tool positioned in {params['reactor_labware']}.{params['target_well']} at {descent_mm:.1f} mm depth")

## ELECTRODEPOSITION - Run In-Well Deposition

Runs the electrochemistry action while the electrode is already positioned in the reactor well.

This cell is the placeholder location for your final deposition logic.

In [ ]:
import csv
import json
import os
from datetime import datetime

import paramiko

# Connection params to Biologic host (Windows host running Biologic stack)
biologic_params = {
    "host": "192.168.0.102",          # SSH host (Windows machine)
    "device_address": "USB0", # Potentiostat address for biologic.connect(...)
    "username": "sdl1_",
    "ssh_port": 22,
    "key_file": None,  # Set to path of private key, or None to auto-detect
    "channel": 4,  # The remote biologic driver uses 1-based channel numbers
    "conda_env": "ot2_workflow",  # Remote conda env to activate before running Python
    "python_executable": "python",  # Python command to run after conda activation
    "eclab_path": None,  # Optional directory containing EClib/blfind DLLs; None uses packaged defaults
    "remote_work_dir": "AC_OTflex_remote",  # Remote folder containing client.py/biologic_host.py/helpers
    "remote_results_dir": None,  # None -> remote_work_dir/raw_electrochemical_data/<date>_expt
    "ocv_check_duration_s": 2,
    "host_start_timeout_s": 45,
}

# Experiment params - files generated by experiment_helper.run_experiment start with experiment_id.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
date_label = datetime.now().strftime("%b%d").lower()
date_number = datetime.now().strftime("%y%m%d")
experiment_params = {
    "experiment_id": "test_1",
    "potential_v": 1.5,
    "step_duration_s": 60,
    "cutoff_capacity_mah_cm2": 10.0,
    "electrode_area_cm2": 1.0,
    "record_every_dt_s": 0.1,
    "record_every_di_a": 0.001,
    "n_cycles": 0,
}

# Create data output directory with timestamp. Downloaded remote CSVs land here.
data_dir = repo_root / "data" / "out" / f"experiment_{timestamp}"
data_dir.mkdir(parents=True, exist_ok=True)

# The `biologic`, `kbio`, `client.py`, `biologic_host.py`, and helper modules are expected
# to exist in this remote conda env / remote_work_dir.

# Find SSH key file (look in common locations)
key_file = biologic_params.get("key_file")
if key_file is None:
    for potential_key in [
        os.path.expanduser("~/.ssh/id_rsa"),
        os.path.expanduser("~/.ssh/id_ed25519"),
        os.path.expanduser("~\\.ssh\\id_rsa"),
        os.path.expanduser("~\\.ssh\\id_ed25519"),
    ]:
        if os.path.exists(potential_key):
            key_file = potential_key
            break

if key_file is None:
    raise RuntimeError("SSH key not found. Set key_file in biologic_params or place key in ~/.ssh/")

print(f"Using SSH key: {key_file}")

# Establish SSH connection to Biologic host
print("\nConnecting to Biologic host via SSH...")
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    ssh.connect(
        biologic_params["host"],
        port=biologic_params["ssh_port"],
        username=biologic_params["username"],
        key_filename=key_file,
        timeout=10,
    )
    print(f"[OK] SSH connected to {biologic_params['username']}@{biologic_params['host']}")
except Exception as e:
    ssh = None
    raise RuntimeError(f"Failed to connect to Biologic host: {e}")

# Store SSH connection in global scope for use in execution cell
_biologic_ssh = ssh

# Remote script: uses client.py / experiment_helper.py in the remote env.
biologic_script = f"""
from pathlib import Path
import os
import subprocess
import sys
import time
import traceback
from datetime import datetime

from biologic import BANDWIDTH, I_RANGE, E_RANGE
from biologic.techniques.ocv import OCVTechnique, OCVParams
from experiment_helper import run_experiment
from client import biologic_stream


def make_ocv(
    rest_time_s=10,
    record_every_dT=0.1,
    record_every_dE=10,
    E_range=E_RANGE.E_RANGE_2_5V,
    bandwidth=BANDWIDTH.BW_5,
):
    return OCVTechnique(OCVParams(
        rest_time_T=rest_time_s,
        record_every_dT=record_every_dT,
        record_every_dE=record_every_dE,
        E_range=E_range,
        bandwidth=bandwidth,
    ))


def run_ocv_connection_check(channel, usb_port):
    check_technique = make_ocv(
        rest_time_s=float({biologic_params['ocv_check_duration_s']}),
        record_every_dT=0.2,
        record_every_dE=10,
        E_range=E_RANGE.E_RANGE_2_5V,
        bandwidth=BANDWIDTH.BW_5,
    )
    points = 0
    for _payload in biologic_stream(
        channel=channel,
        techniques=[check_technique],
        usb_port=usb_port,
    ):
        points += 1
    return points


def start_biologic_host(raw_dir):
    raw_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = raw_dir / "biologic_host_stdout.log"
    stderr_path = raw_dir / "biologic_host_stderr.log"
    stdout_file = open(stdout_path, "ab", buffering=0)
    stderr_file = open(stderr_path, "ab", buffering=0)
    cmd = [sys.executable, "-c", "import biologic_host; biologic_host.main()"]
    kwargs = dict(
        cwd=os.getcwd(),
        stdin=subprocess.DEVNULL,
        stdout=stdout_file,
        stderr=stderr_file,
        close_fds=True,
    )
    if os.name == "nt":
        kwargs["creationflags"] = (
            getattr(subprocess, "DETACHED_PROCESS", 0)
            | getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0)
        )
    process = subprocess.Popen(cmd, **kwargs)
    print("[REMOTE] Started biologic_host.py with PID", process.pid)
    print("[REMOTE] Host stdout log:", stdout_path)
    print("[REMOTE] Host stderr log:", stderr_path)


def ensure_biologic_host_ready(channel, usb_port, raw_dir):
    try:
        points = run_ocv_connection_check(channel, usb_port)
        print("[REMOTE] OCV preflight succeeded; biologic_host.py is reachable. Points:", points)
        return
    except Exception as first_error:
        print("[REMOTE] OCV preflight failed:", first_error)
        print("[REMOTE] Starting biologic_host.py and retrying...")
        start_biologic_host(raw_dir)

    deadline = time.time() + float({biologic_params['host_start_timeout_s']})
    last_error = None
    while time.time() < deadline:
        time.sleep(2)
        try:
            points = run_ocv_connection_check(channel, usb_port)
            print("[REMOTE] OCV preflight succeeded after host start. Points:", points)
            return
        except Exception as retry_error:
            last_error = retry_error
            print("[REMOTE] Waiting for biologic_host.py:", retry_error)

    raise RuntimeError("biologic_host.py did not pass OCV preflight: " + str(last_error))


try:
    # ---- Save path -------------------------
    RAW_DIR = Path("raw_electrochemical_data") / {(date_label + "_expt")!r}
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    # ---- channel / USB port ----------------
    channel = int({biologic_params['channel']})
    usb_port = {biologic_params['device_address']!r}
    expt_id = {experiment_params['experiment_id']!r}

    print("[REMOTE] Python:", sys.executable)
    print("[REMOTE] Working directory:", Path.cwd())
    print("[REMOTE] Result directory:", RAW_DIR.resolve())
    print("[REMOTE] Experiment ID:", expt_id)
    print("[REMOTE] Channel:", channel, "USB port:", usb_port)

    ensure_biologic_host_ready(channel, usb_port, RAW_DIR)

    # --------technique---------------
    ocvTech_10sec = make_ocv(
        rest_time_s=10,
        record_every_dT=0.1,
        record_every_dE=10,
        E_range=E_RANGE.E_RANGE_2_5V,
        bandwidth=BANDWIDTH.BW_5,
    )

    techniques = [ocvTech_10sec]

    # run experiment; experiment_helper writes files named <experiment_id>_*.csv into RAW_DIR
    run_experiment(channel, usb_port, techniques, RAW_DIR, expt_id)
    print("[REMOTE] Experiment complete. Matching files:")
    for path in sorted(RAW_DIR.glob(expt_id + "*")):
        print("[REMOTE_RESULT]", path.name)

except Exception:
    traceback.print_exc()
    raise
"""

print("\n[OK] Experiment setup complete")
print(f"  SSH Host: {biologic_params['host']}")
print(f"  Instrument Address: {biologic_params['device_address']}")
print(f"  Channel: {biologic_params['channel']}")
print(f"  Conda env: {biologic_params['conda_env']}")
print(f"  Python executable: {biologic_params['python_executable']}")
print(f"  Remote work dir: {biologic_params['remote_work_dir']}")
print(f"  Remote results dir: {biologic_params['remote_results_dir'] or f'raw_electrochemical_data/{date_label}_expt'}")
print(f"  Experiment ID / file prefix: {experiment_params['experiment_id']}")
print(f"  Output directory: {data_dir}")


In [ ]:
# Execute the Biologic script on the remote host and download generated result files.
import posixpath

import pandas as pd

print("\n" + "=" * 70)
print("ELECTRODEPOSITION - Biologic Experiment")
print("=" * 70)

# Get SSH connection from setup cell
ssh = None
if "_biologic_ssh" not in globals() or _biologic_ssh is None:
    raise RuntimeError("SSH connection not established. Run the setup cell first.")
ssh = _biologic_ssh


def _remote_abs(path_value, base_dir):
    path_value = str(path_value).replace("\\", "/")
    if path_value.startswith("/") or (len(path_value) > 1 and path_value[1] == ":"):
        return path_value
    return posixpath.join(base_dir, path_value)


def _win_path(path_value):
    return str(path_value).replace("/", "\\")


def _run_remote(cmd, timeout=30):
    stdin, stdout, stderr = ssh.exec_command(cmd, timeout=timeout)
    out = stdout.read().decode(errors="replace").strip()
    err = stderr.read().decode(errors="replace").strip()
    exit_code = stdout.channel.recv_exit_status()
    return exit_code, out, err


try:
    print("\n[1/4] Uploading experiment script to Biologic host...")

    py = biologic_params.get("python_executable", "python")
    conda_env = biologic_params.get("conda_env", "ot2_workflow")
    experiment_id = experiment_params["experiment_id"]
    biologic_params.setdefault("python_executable", py)
    biologic_params.setdefault("conda_env", conda_env)

    # This host is Windows; resolve the remote home without relying on Python first.
    exit_code, home_dir, home_err = _run_remote('cmd.exe /C "echo %USERPROFILE%"', timeout=5)
    home_dir = home_dir.replace("\\", "/")
    if exit_code != 0 or not home_dir or home_dir == "%USERPROFILE%":
        raise RuntimeError(f"Could not resolve remote Windows home directory: {home_err}")

    remote_work_dir = _remote_abs(biologic_params.get("remote_work_dir", "AC_OTflex_remote"), home_dir)
    remote_results_cfg = biologic_params.get("remote_results_dir")
    if remote_results_cfg is None:
        remote_results_dir = posixpath.join(remote_work_dir, "raw_electrochemical_data", f"{date_label}_expt")
    else:
        remote_results_dir = _remote_abs(
            str(remote_results_cfg).format(
                date=date_label,
                date_number=date_number,
                experiment_id=experiment_id,
                timestamp=timestamp,
            ),
            remote_work_dir,
        )

    remote_script_path = posixpath.join(remote_work_dir, "biologic_cycloamperometry_temp.py")
    remote_runner_path = posixpath.join(remote_work_dir, "run_biologic_cycloamperometry_temp.bat")

    for remote_dir in [remote_work_dir, remote_results_dir]:
        code, _, err = _run_remote(
            f'cmd.exe /C if not exist "{_win_path(remote_dir)}" mkdir "{_win_path(remote_dir)}"',
            timeout=10,
        )
        if code != 0:
            raise RuntimeError(f"Could not create remote directory {remote_dir}: {err}")

    runner_script = f"""@echo off
cd /d "{_win_path(remote_work_dir)}"
call conda activate {conda_env}
if errorlevel 1 exit /b %errorlevel%
{py} "%~dp0biologic_cycloamperometry_temp.py"
"""

    sftp = ssh.open_sftp()
    try:
        with sftp.file(remote_script_path, "w") as f:
            f.write(biologic_script)
        with sftp.file(remote_runner_path, "w") as f:
            f.write(runner_script)
    finally:
        sftp.close()
    print(f"[OK] Script uploaded to: {remote_script_path}")
    print(f"[OK] Conda launcher uploaded to: {remote_runner_path}")
    print(f"[OK] Remote result directory: {remote_results_dir}")

    # Execute the script on remote host through a .bat file so conda activation works.
    print("\n[2/4] Executing experiment (this may take several minutes)...")
    print("-" * 70)

    exec_cmd = f'cmd.exe /C "{_win_path(remote_runner_path)}"'
    print(f"Running remote command: {exec_cmd}")
    stdin, stdout, stderr = ssh.exec_command(exec_cmd, timeout=900)

    output_lines = []
    error_lines = []

    for line in stdout:
        line = line.rstrip()
        print(line)
        output_lines.append(line)

    for line in stderr:
        line = line.rstrip()
        if line:
            print(f"[STDERR] {line}")
            error_lines.append(line)

    exit_code = stdout.channel.recv_exit_status()
    print("-" * 70)
    print(f"\n[OK] Remote command completed with exit code: {exit_code}")
    output_text = "\n".join(output_lines)

    print("\n[3/4] Downloading result files... ")
    downloaded_files = []
    sftp = ssh.open_sftp()
    try:
        remote_names = sorted(name for name in sftp.listdir(remote_results_dir) if name.startswith(experiment_id))
        if not remote_names:
            print(f"[WARN] No remote files found with prefix {experiment_id!r} in {remote_results_dir}")
        for name in remote_names:
            remote_path = posixpath.join(remote_results_dir, name)
            local_path = data_dir / name
            sftp.get(remote_path, str(local_path))
            downloaded_files.append(local_path)
            print(f"[OK] Downloaded: {name}")
    finally:
        sftp.close()

    print("\n[4/4] Saving local log, metadata, and display CSV...")
    log_file = data_dir / "experiment_log.txt"
    with open(log_file, "w", encoding="utf-8") as f:
        f.write("=== Biologic Experiment Log ===\n")
        f.write(f"Timestamp: {timestamp}\n")
        f.write(f"Experiment ID: {experiment_id}\n")
        f.write(f"Host: {biologic_params['host']}\n")
        f.write(f"Channel: {biologic_params['channel']}\n")
        f.write(f"Conda env: {conda_env}\n")
        f.write(f"Remote work dir: {remote_work_dir}\n")
        f.write(f"Remote results dir: {remote_results_dir}\n")
        f.write("\n=== STDOUT ===\n")
        f.write(output_text)
        if error_lines:
            f.write("\n=== STDERR ===\n")
            f.write("\n".join(error_lines))
    print(f"[OK] Log file: {log_file}")

    csv_files = [path for path in downloaded_files if path.suffix.lower() == ".csv"]
    csv_file = None
    data_points = 0
    if csv_files:
        frames = []
        for path in csv_files:
            df_part = pd.read_csv(path)
            df_part.insert(0, "source_file", path.name)
            frames.append(df_part)
        df = pd.concat(frames, ignore_index=True, sort=False)

        # Keep the existing display cell working with experiment_helper output names.
        if "Time_s" not in df.columns and "time" in df.columns:
            df["Time_s"] = df["time"]
        if "I_A" not in df.columns and "I" in df.columns:
            df["I_A"] = df["I"]
        if "E_V" not in df.columns and "Ewe" in df.columns:
            df["E_V"] = df["Ewe"]
        if "Capacity_mAh_cm2" not in df.columns and {"Time_s", "I_A"}.issubset(df.columns):
            area = float(experiment_params.get("electrode_area_cm2", 1.0)) or 1.0
            df["Capacity_mAh_cm2"] = (df["I_A"].abs() / area * df["Time_s"] * 1000.0 / 3600.0)

        csv_file = data_dir / "experiment_data.csv"
        df.to_csv(csv_file, index=False)
        data_points = len(df)
        print(f"[OK] Display CSV file: {csv_file}")
        print(f"  Records: {data_points}")
    else:
        print("[WARN] No downloaded CSV files found; display CSV was not created.")

    metadata_file = data_dir / "metadata.json"
    metadata = {
        "experiment_type": "Biologic client experiment",
        "timestamp": timestamp,
        "experiment_id": experiment_id,
        "well": ec_state.get("current_well", "unknown"),
        "biologic_host": biologic_params["host"],
        "biologic_channel": biologic_params["channel"],
        "biologic_conda_env": conda_env,
        "remote_work_dir": remote_work_dir,
        "remote_results_dir": remote_results_dir,
        "experiment_params": experiment_params,
        "downloaded_files": [path.name for path in downloaded_files],
        "data_points": data_points,
        "remote_exit_code": exit_code,
    }
    with open(metadata_file, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"[OK] Metadata file: {metadata_file}")

    try:
        ssh.exec_command(
            f'cmd.exe /C del /Q "{_win_path(remote_script_path)}" "{_win_path(remote_runner_path)}"',
            timeout=5,
        )
    except Exception as cleanup_err:
        print(f"[WARN] Could not clean up remote script: {cleanup_err}")

    if exit_code != 0:
        raise RuntimeError(f"Remote experiment failed with exit code {exit_code}. See {log_file}")
    if not downloaded_files:
        raise RuntimeError(f"Remote experiment finished, but no files beginning with {experiment_id!r} were downloaded.")

    print("\n" + "=" * 70)
    print("[OK] ELECTRODEPOSITION COMPLETE")
    print("=" * 70)
    print(f"\nData saved to: {data_dir}")
    print("\nFiles created/downloaded:")
    print(f"  - {log_file.name}")
    for path in downloaded_files:
        print(f"  - {path.name}")
    if csv_file:
        print(f"  - {csv_file.name}")
    print(f"  - {metadata_file.name}")

except Exception as e:
    print(f"\n[ERROR] Experiment execution failed: {e}")
    import traceback

    traceback.print_exc()

finally:
    # Safely close SSH connection
    if ssh is not None:
        try:
            ssh.close()
            _biologic_ssh = None
            print("\n[OK] SSH connection closed")
        except Exception as close_err:
            print(f"[WARN] Error closing SSH connection: {close_err}")


## ELECTRODEPOSITION - Display Experiment Results

Visualizes the electrochemistry data collected during the CycoAmperometry experiment.

Generates plots of:
- Current vs Time
- Capacity vs Time
- Potential vs Time (if available)


In [ ]:
# Display experiment results from downloaded Biologic CSV files.
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None


def _latest_experiment_dir(out_base):
    exp_dirs = sorted(
        [d for d in out_base.iterdir() if d.is_dir() and d.name != "images"],
        reverse=True,
    )
    if "data_dir" in globals() and Path(data_dir).exists():
        current_dir = Path(data_dir)
        exp_dirs = [current_dir] + [d for d in exp_dirs if d != current_dir]
    return next((d for d in exp_dirs if (d / "metadata.json").exists() or any(d.glob("*.csv"))), None)


def _load_metadata(exp_dir):
    metadata_file = exp_dir / "metadata.json"
    if metadata_file.exists():
        with open(metadata_file, encoding="utf-8") as f:
            return json.load(f)
    return {}


def _raw_csv_paths(exp_dir, metadata):
    downloaded = metadata.get("downloaded_files", [])
    paths = [exp_dir / name for name in downloaded if str(name).lower().endswith(".csv")]
    paths = [path for path in paths if path.exists() and path.name != "experiment_data.csv"]
    if paths:
        return sorted(paths)

    prefix = metadata.get("experiment_id") or globals().get("experiment_params", {}).get("experiment_id")
    if prefix:
        paths = sorted(path for path in exp_dir.glob(f"{prefix}*.csv") if path.name != "experiment_data.csv")
        if paths:
            return paths

    return sorted(path for path in exp_dir.glob("*.csv") if path.name != "experiment_data.csv")


def _parse_result_filename(path, experiment_id=None):
    stem = path.stem
    prefix = experiment_id or ""
    pattern = rf"^{re.escape(prefix)}_(\d+)_(.+)$" if prefix else r"^(.+?)_(\d+)_(.+)$"
    match = re.match(pattern, stem)
    if match and prefix:
        return int(match.group(1)), match.group(2)
    if match:
        return int(match.group(2)), match.group(3)
    return None, "unknown"


def _first_numeric_column(df, candidates):
    lower_to_column = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        col = lower_to_column.get(candidate.lower())
        if col is not None:
            values = pd.to_numeric(df[col], errors="coerce")
            if values.notna().any():
                df[col] = values
                return col
    return None


def _plot_by_file(ax, df, x_col, y_col, title, y_label):
    if not x_col or not y_col:
        ax.text(0.5, 0.5, f"No {y_label} column found", ha="center", va="center")
        ax.set_title(title)
        return
    for source_name, group in df.groupby("source_file", dropna=False):
        plot_df = group[[x_col, y_col]].dropna().sort_values(x_col)
        if plot_df.empty:
            continue
        ax.plot(plot_df[x_col], plot_df[y_col], linewidth=1.5, label=str(source_name))
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if df["source_file"].nunique() > 1:
        ax.legend(fontsize=8)


out_base = repo_root / "data" / "out"
if not out_base.exists():
    print("No experiment data found.")
else:
    exp_dir = _latest_experiment_dir(out_base)
    if exp_dir is None:
        print("No experiment data directories found.")
    else:
        metadata = _load_metadata(exp_dir)
        experiment_id = metadata.get("experiment_id") or globals().get("experiment_params", {}).get("experiment_id")
        raw_paths = _raw_csv_paths(exp_dir, metadata)

        # If only the normalized display copy exists, use it as a fallback.
        csv_paths = raw_paths or ([exp_dir / "experiment_data.csv"] if (exp_dir / "experiment_data.csv").exists() else [])

        print(f"Loading experiment data from: {exp_dir.name}")
        if experiment_id:
            print(f"Experiment ID / file prefix: {experiment_id}")

        if not csv_paths:
            print(f"No CSV files found in: {exp_dir}")
        else:
            frames = []
            for path in csv_paths:
                frame = pd.read_csv(path)
                tech_index, technique = _parse_result_filename(path, experiment_id)
                frame.insert(0, "technique", technique)
                frame.insert(0, "tech_index", tech_index)
                frame.insert(0, "source_file", path.name)
                frames.append(frame)

            df = pd.concat(frames, ignore_index=True, sort=False)
            print(f"Loaded {len(df)} rows from {len(csv_paths)} CSV file(s)")
            print(f"Columns: {', '.join(df.columns)}")

            if display is not None:
                display(df.head(10))
            else:
                print(df.head(10).to_string(index=False))

            time_col = _first_numeric_column(df, ["time", "total_time", "Time_s", "t"])
            current_col = _first_numeric_column(df, ["I", "I_A", "I_avg", "I_mod", "current", "current_A"])
            potential_col = _first_numeric_column(df, ["Ewe", "Ewe_avg", "E_V", "E", "voltage", "Ece"])
            freq_col = _first_numeric_column(df, ["freq", "frequency", "frequency_Hz"])
            phase_col = _first_numeric_column(df, ["phase_Zwe", "phase_Zce", "phase"])

            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            title_id = experiment_id or exp_dir.name
            fig.suptitle(f"Biologic Experiment - {title_id}", fontsize=16, fontweight="bold")

            _plot_by_file(axes[0, 0], df, time_col, potential_col, "Potential vs Time", potential_col or "potential")
            _plot_by_file(axes[0, 1], df, time_col, current_col, "Current vs Time", current_col or "current")

            if freq_col and current_col:
                for source_name, group in df.groupby("source_file", dropna=False):
                    plot_df = group[[freq_col, current_col]].dropna().sort_values(freq_col)
                    if not plot_df.empty:
                        axes[1, 0].semilogx(plot_df[freq_col], plot_df[current_col], marker="o", linewidth=1.2, label=str(source_name))
                axes[1, 0].set_xlabel(freq_col)
                axes[1, 0].set_ylabel(current_col)
                axes[1, 0].set_title("Frequency Sweep")
                axes[1, 0].grid(True, alpha=0.3)
                if df["source_file"].nunique() > 1:
                    axes[1, 0].legend(fontsize=8)
            elif time_col and potential_col:
                for source_name, group in df.groupby("source_file", dropna=False):
                    plot_df = group[[time_col, potential_col]].dropna().sort_values(time_col)
                    if not plot_df.empty:
                        axes[1, 0].scatter(plot_df[time_col], plot_df[potential_col], s=14, label=str(source_name))
                axes[1, 0].set_xlabel(time_col)
                axes[1, 0].set_ylabel(potential_col)
                axes[1, 0].set_title("Potential Samples")
                axes[1, 0].grid(True, alpha=0.3)
            else:
                axes[1, 0].text(0.5, 0.5, "No secondary plot available", ha="center", va="center")
                axes[1, 0].set_title("Secondary Plot")

            axes[1, 1].axis("off")
            file_counts = df.groupby("source_file").size().to_dict()
            file_lines = [f"{name}: {count} rows" for name, count in file_counts.items()]
            summary_text = f"""
EXPERIMENT SUMMARY
{'-' * 40}
Type: {metadata.get('experiment_type', 'Biologic client experiment')}
Timestamp: {metadata.get('timestamp', 'N/A')}
Experiment ID: {experiment_id or 'N/A'}
Well: {metadata.get('well', 'N/A')}
Rows: {len(df)}
CSV Files: {len(csv_paths)}

DETECTED COLUMNS
{'-' * 40}
Time: {time_col or 'not found'}
Potential: {potential_col or 'not found'}
Current: {current_col or 'not found'}
Frequency: {freq_col or 'not found'}
Phase: {phase_col or 'not found'}

FILES
{'-' * 40}
{chr(10).join(file_lines[:8])}
"""
            axes[1, 1].text(
                0.05,
                0.95,
                summary_text,
                transform=axes[1, 1].transAxes,
                fontsize=9,
                verticalalignment="top",
                fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3),
            )

            plt.tight_layout()
            plt.show()

            print("Experiment data visualization complete")


## ELECTRODEPOSITION - Move To Ultrasonic Station A

Relocates the attached electrode tool from reactor position to the first ultrasonic cleaning position.

Use this before triggering the first ultrasonic clean cycle.

In [ ]:
# Move tool to ultrasonic station A (well A1 in B3 sonicator bath)
params = {
    "ultra_slot": ec_defaults["ultra_slot"],
    "sonicator_labware": ec_defaults["sonicator_labware"],
    "target_well": ec_defaults["sonicator_well_a"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "immersion_depth_mm": ec_defaults["sonicator_immersion_mm"],
    "minimum_z_height": 80,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

# Safety first: always raise fully in Z before moving toward the sonicator bath.
import json
import requests

lift_command = {
    "data": {
        "commandType": "moveToAddressableArea",
        "params": {
            "minimumZHeight": int(params["minimum_z_height"]),
            "forceDirect": False,
            "speed": int(params["move_speed"]),
            "pipetteId": rt.oc.pipettes[params["pipette"]]["id"],
            "addressableAreaName": params["ultra_slot"],
            "stayAtHighestPossibleZ": True,
            "offset": {"x": 0.0, "y": 0.0, "z": 0.0},
        },
        "intent": "setup",
    }
}
lift_resp = requests.post(
    url=rt.oc.commandURL,
    headers=rt.oc.headers,
    params={"waitUntilComplete": True},
    data=json.dumps(lift_command),
)
if lift_resp.status_code != 201:
    raise RuntimeError(
        f"Failed to raise toolhead before sonicator move: {lift_resp.status_code} {lift_resp.text}"
    )

bath_lw = rt.lw_ids.get(params["sonicator_labware"], params["sonicator_labware"])
immersion_mm = float(params["immersion_depth_mm"])
if immersion_mm < 0:
    raise ValueError("immersion_depth_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-immersion_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["at_ultrasonic"] = True
ec_state["current_well"] = None
print(f"Tool moved to {params['sonicator_labware']}.{params['target_well']} at {immersion_mm:.1f} mm immersion depth")

## ELECTRODEPOSITION - Ultrasonic Clean A

Triggers the first ultrasonic cleaning cycle while the electrode tool is held at the ultrasonic station.

Use this between deposition cycles to clean the electrode surface.

In [ ]:
# Run ultrasonic transducer (bath A)
params = {
    "channel": 1,
    "duration_s": 15.0,
    "use_auto_off_ms": True,
}

if not ec_state.get("at_ultrasonic", False):
    raise RuntimeError("Tool is not at ultrasonic position. Run a move-to-ultrasonic cell first.")

if ultra is None:
    raise RuntimeError("UltraMQTT is not connected. Run the Connect Devices cell first.")

duration_s = float(params["duration_s"])
if duration_s <= 0:
    raise ValueError("duration_s must be > 0")

channel = int(params["channel"])
duration_ms = max(1, int(round(duration_s * 1000)))

# Match MQTT_Demo behavior (ultra.on()/ultra.off()) and add defensive auto-off.
ultra.ensure_connected()
print(f"Ultrasonic MQTT client={ultra.client_id}, base={ultra.base}, channel={channel}")
if params["use_auto_off_ms"]:
    ultra.on(channel, duration_ms)
else:
    ultra.on(channel)
try:
    await asyncio.sleep(duration_s)
finally:
    # Keep explicit OFF for deterministic shutdown even if auto-off is enabled.
    ultra.off(channel)

print("Ultrasonic run complete.")

## ELECTRODEPOSITION - Move To Ultrasonic Station B

Relocates the attached electrode tool to the second ultrasonic cleaning position.

Use this when running multi-stage cleaning between deposition operations.

In [ ]:
# Move tool to ultrasonic station B (well A2 in B3 sonicator bath)
params = {
    "ultra_slot": ec_defaults["ultra_slot"],
    "sonicator_labware": ec_defaults["sonicator_labware"],
    "target_well": ec_defaults["sonicator_well_b"],
    "offsetX": 0.0,
    "offsetY": 0.0,
    "immersion_depth_mm": ec_defaults["sonicator_immersion_mm"],
    "minimum_z_height": 80,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

# Safety first: always raise fully in Z before moving toward the sonicator bath.
import json
import requests

lift_command = {
    "data": {
        "commandType": "moveToAddressableArea",
        "params": {
            "minimumZHeight": int(params["minimum_z_height"]),
            "forceDirect": False,
            "speed": int(params["move_speed"]),
            "pipetteId": rt.oc.pipettes[params["pipette"]]["id"],
            "addressableAreaName": params["ultra_slot"],
            "stayAtHighestPossibleZ": True,
            "offset": {"x": 0.0, "y": 0.0, "z": 0.0},
        },
        "intent": "setup",
    }
}
lift_resp = requests.post(
    url=rt.oc.commandURL,
    headers=rt.oc.headers,
    params={"waitUntilComplete": True},
    data=json.dumps(lift_command),
)
if lift_resp.status_code != 201:
    raise RuntimeError(
        f"Failed to raise toolhead before sonicator move: {lift_resp.status_code} {lift_resp.text}"
    )

bath_lw = rt.lw_ids.get(params["sonicator_labware"], params["sonicator_labware"])
immersion_mm = float(params["immersion_depth_mm"])
if immersion_mm < 0:
    raise ValueError("immersion_depth_mm must be >= 0")

await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=0.0,
    intSpeed=int(params["move_speed"]),
)
await asyncio.to_thread(
    rt.oc.moveToWell,
    strLabwareName=bath_lw,
    strWellName=params["target_well"],
    strPipetteName=params["pipette"],
    strOffsetStart="top",
    fltOffsetX=float(params["offsetX"]),
    fltOffsetY=float(params["offsetY"]),
    fltOffsetZ=-immersion_mm,
    intSpeed=int(params["move_speed"]),
)

ec_state["at_ultrasonic"] = True
ec_state["current_well"] = None
print(f"Tool moved to {params['sonicator_labware']}.{params['target_well']} at {immersion_mm:.1f} mm immersion depth")

## ELECTRODEPOSITION - Ultrasonic Clean B

Triggers the second ultrasonic cleaning cycle at the second cleaning position.

Use this for additional cleaning before returning to deposition or tool storage.

In [ ]:
# Run ultrasonic transducer (bath B)
params = {
    "channel": 1,
    "duration_s": 15.0,
    "use_auto_off_ms": True,
}

if not ec_state.get("at_ultrasonic", False):
    raise RuntimeError("Tool is not at ultrasonic position. Run a move-to-ultrasonic cell first.")

if ultra is None:
    raise RuntimeError("UltraMQTT is not connected. Run the Connect Devices cell first.")

duration_s = float(params["duration_s"])
if duration_s <= 0:
    raise ValueError("duration_s must be > 0")

channel = int(params["channel"])
duration_ms = max(1, int(round(duration_s * 1000)))

# Match MQTT_Demo behavior (ultra.on()/ultra.off()) and add defensive auto-off.
ultra.ensure_connected()
print(f"Ultrasonic MQTT client={ultra.client_id}, base={ultra.base}, channel={channel}")
if params["use_auto_off_ms"]:
    ultra.on(channel, duration_ms)
else:
    ultra.on(channel)
try:
    await asyncio.sleep(duration_s)
finally:
    # Keep explicit OFF for deterministic shutdown even if auto-off is enabled.
    ultra.off(channel)

print("Ultrasonic run complete.")

## ELECTRODEPOSITION - Return Tool

end electrodeposition

In [ ]:
# Move to another reactor well, or return tool to source
params = {
    "mode": "drop",  # "drop" or "next_well"
    "next_well": "B1",
    "reactor_labware": ec_defaults["reactor_labware"],
    "reactor_offsetX": 0.0,
    "reactor_offsetY": 0.0,
    "descent_at_well_mm": 25.0,
    "source_labware": ec_defaults["source_labware"],
    "source_well": ec_defaults["source_well"],
    "source_offsetX": 0.0,
    "source_offsetY": 0.0,
    "return_dZ": 12.0,
    "move_speed": 80,
    "pipette": ec_defaults["pipette"],
}

if not ec_state.get("tool_attached", False):
    raise RuntimeError("No tool attached. Run the pick-tool cell first.")

rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or getattr(rt, "oc", None) is None:
    raise RuntimeError("OTFlex runtime controller is not available. Run setup/connect cells first.")

if params["mode"] == "next_well":
    dst_lw = rt.lw_ids.get(params["reactor_labware"], params["reactor_labware"])
    descent_mm = float(params["descent_at_well_mm"])

    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=dst_lw,
        strWellName=params["next_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["reactor_offsetX"]),
        fltOffsetY=float(params["reactor_offsetY"]),
        fltOffsetZ=0.0,
        intSpeed=int(params["move_speed"]),
    )
    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=dst_lw,
        strWellName=params["next_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["reactor_offsetX"]),
        fltOffsetY=float(params["reactor_offsetY"]),
        fltOffsetZ=-descent_mm,
        intSpeed=int(params["move_speed"]),
    )
    ec_state["current_well"] = params["next_well"]
    ec_state["at_ultrasonic"] = False
    print(f"Tool moved to next well {params['reactor_labware']}.{params['next_well']}")
else:
    src_lw = rt.lw_ids.get(params["source_labware"], params["source_labware"])

    await asyncio.to_thread(
        rt.oc.moveToWell,
        strLabwareName=src_lw,
        strWellName=params["source_well"],
        strPipetteName=params["pipette"],
        strOffsetStart="top",
        fltOffsetX=float(params["source_offsetX"]),
        fltOffsetY=float(params["source_offsetY"]),
        fltOffsetZ=10,
        intSpeed=100,
    )
    await asyncio.to_thread(
        rt.oc.dropTip,
        strPipetteName=params["pipette"],
        boolDropInDisposal=False,
        strLabwareName=src_lw,
        strWellName=params["source_well"],
        strOffsetStart="bottom",
        fltOffsetX=float(params["source_offsetX"]),
        fltOffsetY=float(params["source_offsetY"]),
        fltOffsetZ=float(params["return_dZ"]),
    )

    ec_state["tool_attached"] = False
    ec_state["current_well"] = None
    ec_state["at_ultrasonic"] = False
    print(f"Tool dropped back at {params['source_labware']}.{params['source_well']}")

## Flush Tool Wells (otflexFlushWell)

Flushes wells `A1` and `B1` using pump IDs configured in params.
Each well flush now explicitly starts with OUT pump and ends with OUT pump.

In [ ]:
# Action params (editable)
params = {
    "from": {"labware": "electrode_module", "well": "C1", "offsetX": 0.0, "offsetY": 0.0, "offsetZ": 0.0},
    "to": {"labware": "auto_reactor", "well": ["A1", "B1"], "offsetX": 0, "offsetY": 0, "offsetZ": 5.0},
    "pipette": "p1000_single_flex",
    "in_pump_id": 2,
    "out_pump_id": 3,
    "in_time_ms": 2000,
    "out_time_ms": 5000,
    "repeats": 4,
    "purge_ms": 0,
    "return_dZ": 12.0,
}

# Preflight: flushWell relies on OTFlex runtime MQTT pump handle.
rt = getattr(getattr(otflex, "mod", None), "_RT", None)
if rt is None or rt.mqtt_pumps is None:
    raise RuntimeError(
        "OTFlex runtime MQTT pumps are not configured. Run the Connect Devices cell first."
    )

await otflex.flushWell(params)
print(
    f"Flush action complete (order per well: OUT -> (IN/OUT)x{params['repeats']} -> OUT; "
    f"in pump {params['in_pump_id']} for {params['in_time_ms']}ms, "
    f"out pump {params['out_pump_id']} for {params['out_time_ms']}ms)."
)

## Raise Autoreactor Final (mqttReactor)

Final reactor raise before furnace sequence.

In [ ]:
# Action params (editable)
params = {
    "direction": "forward",
    "duration_ms": 5000,
}

if params["direction"].lower() == "forward":
    await asyncio.to_thread(reactor.forward, params["duration_ms"])
else:
    await asyncio.to_thread(reactor.reverse, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Reactor raise action complete.")

## Open Furnace Door (mqttFurnace)

Publishes `OPEN:6400` and waits to stay aligned with movement duration.

In [ ]:
# Action params (editable)
params = {
    "action": "open",
    "duration_ms": 6400,
}

if params["action"].lower() == "open":
    await asyncio.to_thread(furnace.open, params["duration_ms"])
else:
    await asyncio.to_thread(furnace.close, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Furnace open action complete.")

## Run xArm Reactor-to-Furnace Transfer

This action replaces the fixed post-open pause.

The notebook waits until `src/core/reactor2furnace.py` completes, then the next action can close the furnace door.

In [ ]:
# Action params (editable)
params = {
    "script_relpath": "src/core/reactor2furnace.py",
    "python_exe": sys.executable,
    "timeout_s": 0,  # 0 means no timeout
}

script_path = (repo_root / params["script_relpath"]).resolve()
if not script_path.exists():
    raise FileNotFoundError(f"xArm script not found: {script_path}")

cmd = [params["python_exe"], str(script_path)]
print("Running xArm transfer script:", " ".join(cmd))

run_kwargs = dict(
    cwd=str(repo_root),
    text=True,
    capture_output=True,
    check=False,
    timeout=None if int(params["timeout_s"]) <= 0 else float(params["timeout_s"]),
)

try:
    result = subprocess.run(cmd, **run_kwargs)
except subprocess.TimeoutExpired as exc:
    raise TimeoutError(
        f"xArm transfer timed out after {params['timeout_s']}s: {script_path}"
    ) from exc

if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"xArm transfer failed with exit code {result.returncode}: {script_path}"
    )

print("xArm reactor-to-furnace action complete.")

## Close Furnace Door (mqttFurnace)

Publishes `CLOSE:10000` and waits to align with door motion.

In [ ]:
# Action params (editable)
params = {
    "action": "close",
    "duration_ms": 10000,
}

if params["action"].lower() == "open":
    await asyncio.to_thread(furnace.open, params["duration_ms"])
else:
    await asyncio.to_thread(furnace.close, params["duration_ms"])
await asyncio.sleep(max(0.0, float(params["duration_ms"]) / 1000.0))
print("Furnace close action complete.")

## End Marker

No hardware action. This marks the workflow end.

In [ ]:
print("Workflow sequence reached end marker.")

## Teardown - Safe Shutdown

Run this at the end, or immediately if any step errors.

This mirrors the runner's best-effort shutdown intent:
- turn off active MQTT outputs
- disconnect MQTT clients and beacon
- disconnect OTFlex

In [ ]:
try:
    _best_effort_all_off(pumps=pumps, ultra=ultra, heat=heat, reactor=reactor, furnace=furnace)
except Exception as exc:
    print(f"Best-effort OFF warning: {exc}")

for dev, name in ((pumps, "pumps"), (ultra, "ultra"), (heat, "heat"), (reactor, "reactor"), (furnace, "furnace")):
    if dev is not None:
        try:
            dev.disconnect()
        except Exception as exc:
            print(f"Disconnect warning ({name}): {exc}")

if beacon is not None:
    try:
        beacon.stop()
    except Exception as exc:
        print(f"Beacon stop warning: {exc}")

await otflex.disconnect()
print("Teardown complete.")